# ARENAS — Adjacency Matrix Explorer
**2 files needed:** adjacency `.pkl` + dataset `.csv`  
**How to run:** `Kernel → Restart & Run All`, then set file paths below.

In [1]:
import subprocess, sys
for p in ['pyvis', 'ipywidgets', 'networkx', 'pandas', 'numpy', 'matplotlib']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=False)

import pickle, io, os, threading, warnings
import numpy as np
import pandas as pd
import networkx as nx
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import display, HTML, clear_output
from pyvis.network import Network
warnings.filterwarnings('ignore')

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('✔ Ready')

✔ Ready


In [2]:
#ADJ_PATH = '/data/ARENAS_Automatic_Extremist_Analysis/ARENAS_Automatic_Extremist_Analysis/DL_approaches/ARENAS-Context-Aware-Graph-Structure-Learning-ContextGSL-/adjacency_matrices/round2/adj_final__exp4__max10__K5__1772558890__adjacency_final__exp4__ctxcols_In-Group_Out-group__ntrials_1.pkl'   # ← path to your .pkl adjacency matrix

# ← path to your .pkl adjacency matrix
ADJ_PATH = 'max10_adj.pkl' 

# ← path to your dataset .csv
CSV_PATH = 'arenas_french.csv'     

if IN_COLAB:
    from google.colab import files
    print('⬆️  Upload adjacency .pkl')
    up  = files.upload()
    RAW_ADJ = pickle.loads(list(up.values())[0])
    print('⬆️  Upload dataset .csv')
    up2 = files.upload()
    NODE_DF = pd.read_csv(io.BytesIO(list(up2.values())[0]), index_col=0)
else:
    with open(ADJ_PATH, 'rb') as f:
        RAW_ADJ = pickle.load(f)
    NODE_DF = pd.read_csv(CSV_PATH, index_col=0)

if hasattr(RAW_ADJ, 'numpy'):
    RAW_ADJ = RAW_ADJ.numpy()
RAW_ADJ = np.array(RAW_ADJ, dtype=np.float32)
np.fill_diagonal(RAW_ADJ, 0)
NODE_DF = NODE_DF.reset_index(drop=True)

# Normalize string labels: strip whitespace and trailing/leading punctuation
# so that e.g. "Heterosexual People." == "Heterosexual People"
# and "Anti-Gypsyism," == "Anti-Gypsyism"
for _nc in NODE_DF.columns:
    if NODE_DF[_nc].dtype == object:
        NODE_DF[_nc] = (
            NODE_DF[_nc]
            .str.strip()
            .str.strip('.,;:')
            .str.strip()
        )

NODE_ATTRS = NODE_DF.to_dict('index')
N          = RAW_ADJ.shape[0]
COLS       = list(NODE_DF.columns)
CAT_COLS   = [c for c in COLS if NODE_DF[c].dtype == object or NODE_DF[c].nunique() <= 20]

print(f'✔ Matrix : {RAW_ADJ.shape}')
print(f'  Dataset: {N} rows')
print(f'  Columns: {COLS}')

✔ Matrix : (791, 791)
  Dataset: 791 rows
  Columns: ['Text', 'Topic', 'Tone of Post', 'In-Group', 'Out-group', 'Narrator', 'Intolerance', 'Superiority of in-group', 'Hostility to out-group (e.g. verbal attacks, belittlement, instillment of fear, incitement to violence)', 'Polarization/Othering', 'Perceived Threat', 'Character(s)', 'Setting', 'Initiating Problem', 'Emotional response', 'Solution', 'Appeal to Authority', 'Appeal to Reason', 'Appeal to Probability', 'Conspiracy Theories', 'Irony/Humor']


In [3]:
def get_edges(t):
    mask = np.triu(RAW_ADJ >= t, k=1)
    r, c = np.where(mask)
    return r, c, RAW_ADJ[r, c]

def make_graph(t):
    r, c, w = get_edges(t)
    G = nx.Graph()
    G.add_nodes_from(range(N))
    G.add_weighted_edges_from(zip(r.tolist(), c.tolist(), w.tolist()))
    nx.set_node_attributes(G, NODE_ATTRS)
    return G

def to_hex(cmap_name, val, vmin, vmax):
    r, g, b, _ = cm.get_cmap(cmap_name)((val - vmin) / (vmax - vmin + 1e-9))
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

def show(fig):
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print('✔ Helpers ready')

✔ Helpers ready


---
## Section 1 — Raw graph as constructed

This section plots the graph as it was constructed

- Use **Colour by** to highlight node categories.  
- Use **Max nodes** to focus on the highest-degree nodes.  
- Toggle **Show node IDs** to overlay each node's index directly on the graph (auto-hidden above 300 nodes for readability).

In [4]:
# =============================================================================
# Section 0 — Raw Graph Visualisation
# =============================================================================

# -----------------------------------------------------------------------------
# Display constants
# -----------------------------------------------------------------------------
S0_MAX_ID_NODES = 300
S0_FONT_MAX     = 18
S0_FONT_MIN     = 10
S0_FONT_SCALE   = 350
S0_FONT_NMIN    = 12


# -----------------------------------------------------------------------------
# Widgets
# -----------------------------------------------------------------------------
col_select = widgets.Dropdown(
    options=['(none)'] + CAT_COLS,
    value='(none)',
    description='Colour by:',
    layout=widgets.Layout(width='260px'),
    style={'description_width': 'initial'}
)

max_nodes_slider = widgets.IntSlider(
    value=300,
    min=10,
    max=N,
    step=10,
    description='Max nodes:',
    layout=widgets.Layout(width='380px'),
    style={'description_width': 'initial'}
)

label_filter = widgets.SelectMultiple(
    options=[],
    value=[],
    description='Filter labels:',
    layout=widgets.Layout(width='260px', height='120px'),
    style={'description_width': 'initial'}
)

show_ids_box = widgets.Checkbox(
    value=True,
    description='Show node IDs'
)

draw_button = widgets.Button(
    description='Draw raw graph',
    button_style='primary',
    layout=widgets.Layout(width='150px')
)

output_area = widgets.Output()


# -----------------------------------------------------------------------------
# Update label filter
# -----------------------------------------------------------------------------
def update_labels(change):

    column = change['new']

    if column == '(none)':
        label_filter.options = []
        label_filter.value = []
        return

    values = sorted(
        NODE_DF[column].dropna().unique().tolist(),
        key=str
    )

    label_filter.options = values
    label_filter.value = values


col_select.observe(update_labels, names='value')


# -----------------------------------------------------------------------------
# Draw graph
# -----------------------------------------------------------------------------
def draw_graph(_):

    with output_area:

        clear_output(wait=True)

        colour_col = col_select.value
        max_nodes  = max_nodes_slider.value
        show_ids   = show_ids_box.value

        selected_labels = (
            list(label_filter.value)
            if colour_col != '(none)'
            else None
        )

        # ---------------------------------------------------------------------
        # Edge list
        # ---------------------------------------------------------------------
        rows, cols = np.nonzero(np.triu(RAW_ADJ, k=1))

        # ---------------------------------------------------------------------
        # Node filtering
        # ---------------------------------------------------------------------
        if colour_col != '(none)' and selected_labels is not None:

            node_mask = NODE_DF[colour_col].isin(selected_labels)
            valid_nodes = set(NODE_DF.index[node_mask])

        else:
            valid_nodes = set(range(N))

        # ---------------------------------------------------------------------
        # Select top nodes by degree
        # ---------------------------------------------------------------------
        degree = np.array((RAW_ADJ > 0).sum(axis=1))

        ranked_nodes = sorted(
            valid_nodes,
            key=lambda n: -degree[n]
        )

        subset = ranked_nodes[:max_nodes]
        subset_set = set(subset)
        subset_arr = np.array(subset)

        # ---------------------------------------------------------------------
        # Filter edges
        # ---------------------------------------------------------------------
        edge_mask = (
            np.isin(rows, subset_arr) &
            np.isin(cols, subset_arr)
        )

        sub_rows = rows[edge_mask]
        sub_cols = cols[edge_mask]

        n_nodes = len(subset_set)
        n_edges = len(sub_rows)

        print(f"sub_rows: {sub_rows}")
        
            
        print(f'Nodes: {n_nodes} | Edges: {n_edges} (no threshold)')

        # ---------------------------------------------------------------------
        # Build graph
        # ---------------------------------------------------------------------
        G = nx.Graph()
        G.add_nodes_from(subset_set)
        G.add_edges_from(zip(sub_rows, sub_cols))

        nx.set_node_attributes(G, NODE_ATTRS)

        node_list = list(G.nodes())

        if not node_list:
            print("No nodes to display.")
            return

        # ---------------------------------------------------------------------
        # Layout
        # ---------------------------------------------------------------------
        if n_nodes <= 80:
            pos = nx.kamada_kawai_layout(G)
        else:
            pos = nx.spring_layout(
                G,
                seed=42,
                k=2.5 / np.sqrt(max(n_nodes, 1)),
                iterations=60
            )

        # ---------------------------------------------------------------------
        # Node sizes
        # ---------------------------------------------------------------------
        deg = np.array([G.degree(n) for n in node_list])

        dmin, dmax = deg.min(), deg.max()

        node_sizes = 80 + 420 * (deg - dmin) / (dmax - dmin + 1e-9)

        # ---------------------------------------------------------------------
        # Node colours
        # ---------------------------------------------------------------------
        palette = plt.get_cmap("tab20b")

        if colour_col != "(none)":

            uniq = sorted(
                NODE_DF[colour_col].dropna().unique().tolist(),
                key=str
            )

            cat_index = {v: i for i, v in enumerate(uniq)}

            node_colors = [
                palette(cat_index.get(NODE_DF[colour_col].iloc[n], 0) % 20)
                for n in node_list
            ]

        else:

            cmap = plt.get_cmap("plasma")

            node_colors = [
                cmap(
                    0.15 + 0.75 *
                    (G.degree(n) - dmin) / (dmax - dmin + 1e-9)
                )
                for n in node_list
            ]

        # ---------------------------------------------------------------------
        # Plot (Satellite style)
        # ---------------------------------------------------------------------
        #fig, ax = plt.subplots(figsize=(13,10))
        fig, ax = plt.subplots(figsize=(23,20))

        fig.patch.set_facecolor("#0b0f1a")
        ax.set_facecolor("#0b0f1a")

        # Edges
        nx.draw_networkx_edges(
            G,
            pos,
            ax=ax,
            alpha=0.55,
            width=1.3,
            edge_color="#8FA8FF"
        )


        # Node glow
        nx.draw_networkx_nodes(
            G,
            pos,
            node_size=node_sizes * 1.8,
            node_color=node_colors,
            alpha=0.15,
            linewidths=0
        )

        # Nodes
        nx.draw_networkx_nodes(
            G,
            pos,
            node_color=node_colors,
            node_size=node_sizes,
            edgecolors="white",
            linewidths=0.7,
            alpha=0.95
        )

        # ---------------------------------------------------------------------
        # Label placement
        # ---------------------------------------------------------------------
        if show_ids and n_nodes <= S0_MAX_ID_NODES:

            font_size = max(
                S0_FONT_MIN,
                min(
                    S0_FONT_MAX,
                    int((S0_FONT_SCALE) / max(n_nodes, S0_FONT_NMIN))
                )
            )

            center_x = np.mean([p[0] for p in pos.values()])
            center_y = np.mean([p[1] for p in pos.values()])

            for node in node_list:

                x, y = pos[node]

                dx = x - center_x
                dy = y - center_y

                norm = np.sqrt(dx**2 + dy**2) + 1e-9

                dx /= norm
                dy /= norm

                offset = 0.07

                lx = x + dx * offset
                ly = y + dy * offset

                # connector line
                ax.plot(
                    [x, lx],
                    [y, ly],
                    color="#9aa4c7",
                    linewidth=0.6,
                    alpha=0.7
                )

                # label
                ax.text(
                    lx,
                    ly,
                    str(node),
                    fontsize=font_size,
                    color="white",
                    ha="center",
                    va="center",
                    bbox=dict(
                        boxstyle="round,pad=0.3",
                        fc="#111827",
                        ec="#6b7280",
                        lw=0.6,
                        alpha=0.95
                    )
                )

        # ---------------------------------------------------------------------
        # Legend / Colorbar
        # ---------------------------------------------------------------------
        if colour_col != "(none)":

            shown = sorted(
                {NODE_DF[colour_col].iloc[n] for n in node_list},
                key=str
            )

            handles = [
                plt.Line2D(
                    [0], [0],
                    marker="o",
                    color="w",
                    markerfacecolor=palette(cat_index[v] % 20),
                    markersize=8,
                    label=str(v)
                )
                for v in shown
            ]

        
            
            
            
            
            ax.legend(
                handles=handles,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=True,
                fontsize=12,
                title=colour_col,
                title_fontsize=16,
                labelspacing=1.2,
                borderpad=1.1
            )
            
            
            
            
            
            colour_title = f"coloured by {colour_col}"

        else:

            sm = plt.cm.ScalarMappable(
                cmap="plasma",
                norm=plt.Normalize(vmin=dmin, vmax=dmax)
            )

            sm.set_array([])

            cbar = plt.colorbar(sm, ax=ax)
            cbar.set_label("Degree")

            colour_title = "coloured by degree"

        # ---------------------------------------------------------------------
        # Title
        # ---------------------------------------------------------------------
        ax.set_title(
            f"Raw graph — {colour_title}\n"
            f"{n_nodes} nodes · {n_edges} edges",
            fontsize=14,
            color="white"
        )

        ax.axis("off")

        plt.tight_layout(rect=[0,0,0.82,1])
        plt.show()


draw_button.on_click(draw_graph)


# -----------------------------------------------------------------------------
# Display UI
# -----------------------------------------------------------------------------
display(
    widgets.HBox([col_select, max_nodes_slider]),
    widgets.HBox([label_filter, show_ids_box]),
    draw_button,
    output_area
)

Button(button_style='primary', description='Draw raw graph', layout=Layout(width='150px'), style=ButtonStyle()…

Output()

---
## Section 2 — Node inspector

Select a **threshold** and a **Node ID**, then click **Inspect node** to see:

- A styled info panel with the node's degree and all its attributes.  
- A neighbourhood sub-graph showing all direct neighbours with edge-weight bars and every node ID labelled.

In [5]:
# ── 4 — Node inspector (raw — all neighbours, no threshold, no k limit) ──────

node_input = widgets.BoundedIntText(
    value=0, min=0, max=N-1, description='Node ID:',
    style={'description_width': '80px'}, layout=widgets.Layout(width='200px')
)
insp_btn = widgets.Button(description='Inspect', button_style='info')
insp_out = widgets.Output()

def inspect(_):
    with insp_out:
        clear_output(wait=True)
        nid = node_input.value

        # ── Feature table ─────────────────────────────────────────────────
        row = NODE_DF.iloc[nid]
        rows_html = ''.join(
            f'<tr>'
            f'<td style="font-weight:bold;background:#f5f5f5;padding:3px 10px">{col}</td>'
            f'<td style="padding:3px 10px">{str(row[col])[:200]}</td>'
            f'</tr>'
            for col in COLS
        )
        display(HTML(
            f'<h4>Node {nid}</h4>'
            f'<table style="border-collapse:collapse;font-size:13px">'
            f'<tr>'
            f'<th style="background:#333;color:white;padding:4px 10px">Feature</th>'
            f'<th style="background:#333;color:white;padding:4px 10px">Value</th>'
            f'</tr>'
            f'{rows_html}</table>'
        ))

        # ── ALL raw neighbours — no threshold, no k cap ───────────────────
        ew = RAW_ADJ[nid].copy()
        ew[nid] = 0  # exclude self-loop

        all_nbrs = np.argsort(ew)[::-1]
        all_nbrs = all_nbrs[ew[all_nbrs] > 0]  # only real edges

        degree = len(all_nbrs)
        print(f'Raw degree (all edges, no threshold): {degree}')

        if degree == 0:
            print('No neighbours found.')
            return

        # ── Bar chart of all neighbours ───────────────────────────────────
        fig, ax = plt.subplots(figsize=(min(degree * 0.5 + 2, 16), 3.5))
        ax.bar([str(i) for i in all_nbrs],
               [float(ew[i]) for i in all_nbrs],
               color='mediumseagreen', edgecolor='white', linewidth=0.4)
        ax.set_title(f'All {degree} neighbours of node {nid} (raw)', fontsize=12)
        ax.set_xlabel('Neighbour node ID')
        ax.set_ylabel('Edge weight')
        ax.tick_params(axis='x', rotation=90)
        show(fig)

        # ── Neighbour table ───────────────────────────────────────────────
        text_col = next(
            (c for c in ['Text', 'text', 'tweet', 'sentence'] if c in COLS), None
        )
        nbr_rows = ''.join(
            f'<tr>'
            f'<td style="padding:3px 8px">{i}</td>'
            f'<td style="padding:3px 8px">{ew[i]:.4f}</td>'
            f'<td style="padding:3px 8px">'
            f'{str(NODE_DF.iloc[i][text_col])[:120] if text_col else "—"}'
            f'</td></tr>'
            for i in all_nbrs
        )
        display(HTML(
            f'<table style="border-collapse:collapse;font-size:12px;margin-top:8px">'
            f'<tr style="background:#333;color:white">'
            f'<th style="padding:4px 8px">Node</th>'
            f'<th style="padding:4px 8px">Weight</th>'
            f'<th style="padding:4px 8px">{text_col or "(no text col)"}</th>'
            f'</tr>{nbr_rows}</table>'
        ))

insp_btn.on_click(inspect)
display(widgets.HBox([node_input, insp_btn]))
display(insp_out)

Output()